# Trace walkthrough: reading an agent trace programmatically

The Jaeger UI is the right tool for exploring one trace; the HTTP API is the right
tool for asking questions across traces. This notebook runs one diagnosis, pulls its
trace from Jaeger, and answers three questions the UI can't answer at a glance:

1. Where did the wall-clock time go — model, tools, or overhead?
2. What did each tool call actually return to the model?
3. What does token usage look like per model call, including the extraction pass?

**Prerequisites:** `docker compose --profile jaeger up -d` and a running Ollama with
`qwen3` pulled. Run from the repo root: `uv run --with jupyter jupyter lab`.

In [1]:
import json
import time
import urllib.request

from brewtrace.agent import build_agent, run_diagnosis
from brewtrace.models import parse_brew_log
from brewtrace.telemetry import brew_request_span, setup_telemetry

JAEGER = "http://localhost:16686"
BREW_TEXT = "16g/250g V60, 94C, 3:45 drawdown, sour and thin"

setup_telemetry(otlp=True)

## Run one traced diagnosis

Same code path as the CLI: parse deterministically, open the `brewtrace.request`
parent span with the parsed `brew.*` attributes, run the agent inside it.

In [2]:
brew = parse_brew_log(BREW_TEXT)
agent = build_agent()

with brew_request_span(brew, extra={"notebook.run": True}) as span:
    trace_id = format(span.get_span_context().trace_id, "032x")
    advice, result = run_diagnosis(agent, BREW_TEXT)

print(f"advice: change {advice.variable.value} -> {advice.direction.value}")
print(f"trace id: {trace_id}")

advice: change grind -> finer
trace id: c66f6432515f3cf7442c257285ca1663


> Note: the structured *direction* occasionally comes back as `increase` when the
> prose said "finer" — the extraction-mapping failure bucket measured in the eval
> baseline (see `docs/tutorial.md`). The prose answer and the `recommend_adjustment`
> tool span in the trace below always show the ground truth.

## Fetch the trace from Jaeger

The exporter batches spans, so give it a moment to flush, then request the exact
trace by id.

In [3]:
def fetch_trace(trace_id: str, attempts: int = 15) -> dict:
    # Wait until the ROOT span has flushed: it ends last but the batch
    # exporter may deliver the child spans first.
    for _ in range(attempts):
        try:
            with urllib.request.urlopen(f"{JAEGER}/api/traces/{trace_id}") as resp:
                data = json.load(resp)
            candidates = data.get("data", [])
            if candidates and any(
                s["operationName"] == "brewtrace.request" for s in candidates[0]["spans"]
            ):
                return candidates[0]
        except urllib.error.HTTPError:
            pass
        time.sleep(2)
    raise RuntimeError("trace not found; is Jaeger up and the exporter flushing?")

trace = fetch_trace(trace_id)
spans = trace["spans"]
print(f"{len(spans)} spans")

14 spans


## The span tree, with durations

In [4]:
by_id = {s["spanID"]: s for s in spans}
children = {}
roots = []
for s in spans:
    parents = [r["spanID"] for r in s["references"] if r["refType"] == "CHILD_OF"]
    if parents and parents[0] in by_id:
        children.setdefault(parents[0], []).append(s)
    else:
        roots.append(s)

def show(span, depth=0):
    ms = span["duration"] / 1000
    print(f"{'  ' * depth}{span['operationName']:<45} {ms:>10.0f} ms")
    for child in sorted(children.get(span["spanID"], []), key=lambda s: s["startTime"]):
        show(child, depth + 1)

for root in sorted(roots, key=lambda s: s["startTime"]):
    show(root)

brewtrace.request                                  80963 ms
  invoke_agent Strands Agents                        66003 ms
    execute_event_loop_cycle                           56910 ms
      chat                                               56900 ms
      execute_tool calculate_brew_ratio                      5 ms
      execute_tool assess_drawdown_time                      4 ms
      execute_tool retrieve_recipe_notes                     5 ms
      execute_tool recommend_adjustment                      3 ms
    execute_event_loop_cycle                            9091 ms
      chat                                                9090 ms
  invoke_agent Strands Agents                        14955 ms
    execute_event_loop_cycle                           14951 ms
      chat                                               14950 ms
      execute_tool BrewAdvice                                0 ms


## Where did the time go?

Sum leaf-level model (`chat`) and tool (`execute_tool *`) spans against the root.
The gap between the root duration and the sum is orchestration overhead plus the
structured-output extraction pass (its `chat` span lives in a second trace).

In [5]:
root = next(s for s in spans if s["operationName"] == "brewtrace.request")
total_ms = root["duration"] / 1000
model_ms = sum(s["duration"] for s in spans if s["operationName"] == "chat") / 1000
tool_spans = [s for s in spans if s["operationName"].startswith("execute_tool")]
tool_ms = sum(s["duration"] for s in tool_spans) / 1000

print(f"total request      {total_ms:>10.0f} ms")
print(f"model calls        {model_ms:>10.0f} ms  ({model_ms / total_ms:.0%})")
print(f"tool executions    {tool_ms:>10.0f} ms  ({tool_ms / total_ms:.0%})")

total request           80963 ms
model calls             80940 ms  (100%)
tool executions            18 ms  (0%)


The lesson generalizes: local-agent latency is model inference. Tools are
microseconds; everything worth optimizing is in the `chat` spans.

## What did the tools return?

Strands records tool inputs on the `execute_tool` spans; results are in the tool
messages. The span attributes alone already show the call arguments the model chose.

In [6]:
for s in sorted(tool_spans, key=lambda s: s["startTime"]):
    tags = {t["key"]: t["value"] for t in s["tags"]}
    name = s["operationName"].removeprefix("execute_tool ")
    print(f"{name:<28} {s['duration'] / 1000:>6.1f} ms   id={tags.get('gen_ai.tool.call.id', '?')}")

calculate_brew_ratio            5.1 ms   id=tooluse_58376120ed0e43d29387a6ef
assess_drawdown_time            4.0 ms   id=tooluse_3b7534e1048543948a39aef5
retrieve_recipe_notes           5.5 ms   id=tooluse_c9fcb07eb53d428ab94f3c29
recommend_adjustment            3.0 ms   id=tooluse_b34db838f20f4561ad6a0446
BrewAdvice                      0.4 ms   id=tooluse_cb65b17f9cfe46f59a53980b


## Token accounting per model call

In [7]:
for s in sorted(spans, key=lambda s: s["startTime"]):
    if s["operationName"] != "chat":
        continue
    tags = {t["key"]: t["value"] for t in s["tags"]}
    print(
        f"chat  in={tags.get('gen_ai.usage.input_tokens', '?'):>6}"
        f"  out={tags.get('gen_ai.usage.output_tokens', '?'):>6}"
        f"  ttft={tags.get('gen_ai.server.time_to_first_token', '?')} ms"
    )
print()
usage = result.metrics.accumulated_usage
print(f"agent-loop total: in={usage['inputTokens']}, out={usage['outputTokens']}")

chat  in=   836  out=  1990  ttft=4 ms
chat  in=  1473  out=   314  ttft=7 ms
chat  in=   447  out=   534  ttft=6 ms

agent-loop total: in=2309, out=2304


## Root-span attributes: the queryable index

These come from the deterministic parser, not the model — which is what makes traces
searchable in Jaeger (`taste.primary_defect=sour_thin`, `eval.passed=false`) no matter
what the agent did.

In [8]:
for t in sorted(root["tags"], key=lambda t: t["key"]):
    if t["key"].startswith(("brew.", "taste.", "notebook.")):
        print(f"{t['key']:<28} {t['value']}")

brew.dose_g                  16
brew.drawdown_seconds        225
brew.method                  v60
brew.ratio                   15.62
brew.temperature_c           94
brew.water_g                 250
notebook.run                 True
taste.primary_defect         sour_thin
